# 05 — Temporal Trends and Emerging Topics

Analiza producción anual, evolución temática, antigüedad promedio de keywords y términos emergentes.

> Notebook reproducible del Paper Semantic Bibliometric Toolkit v1.0.0. No elimina documentos ni convierte resultados exploratorios en conclusiones sin revisión manual.

## 1. Cargar corpus, keywords y temas

In [ ]:

from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = None
current = Path.cwd().resolve()
for candidate in [current] + list(current.parents):
    if (candidate / "config.json").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró config.json.")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from core import (
    find_project_root, read_json, load_config, ensure_directories,
    load_scopus_csv, load_clean_corpus, normalize_spacing,
    normalize_identifier, normalize_title, normalize_keyword,
    split_semicolon, frequency_table, save_csv, save_json,
    markdown_table, combined_text
)

CONFIG = load_config(PROJECT_ROOT)
ensure_directories(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = PROJECT_ROOT / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
THESAURUS_DIR = PROJECT_ROOT / "thesaurus"
VALIDATION_DIR = PROJECT_ROOT / "validation"

print("Project root:", PROJECT_ROOT)
print("Toolkit version:", CONFIG.get("toolkit_version"))
print("Configured corpus:", PROJECT_ROOT / CONFIG["input_csv"])

df = load_clean_corpus(PROJECT_ROOT)
author_kw = pd.read_csv(OUTPUT_DIR / "03_author_keywords_exploded.csv", encoding="utf-8-sig")
theme_docs = pd.read_csv(OUTPUT_DIR / "04_theme_document_matches.csv", encoding="utf-8-sig")


## 2. Producción anual

In [ ]:

df["Year numeric"] = pd.to_numeric(df["Year"], errors="coerce")
annual = df.dropna(subset=["Year numeric"]).assign(Year=lambda x: x["Year numeric"].astype(int)).groupby("Year")["EID"].nunique().rename("Documents").reset_index().sort_values("Year")
annual["Year status"] = annual["Year"].map(lambda year: "partial/future" if int(year) in CONFIG.get("partial_or_future_years", []) else "complete")
save_csv(annual, OUTPUT_DIR / "05_documents_by_year.csv")
display(annual.tail(20))


## 3. Evolución de keywords

In [ ]:

author_kw["Year"] = pd.to_numeric(author_kw["Year"], errors="coerce")
keyword_year = author_kw.dropna(subset=["Year"]).drop_duplicates(["EID", "Keyword normalized"]).groupby(["Keyword normalized", "Year"])["EID"].nunique().rename("Documents").reset_index()
keyword_summary = keyword_year.groupby("Keyword normalized").apply(lambda group: pd.Series({
    "Total documents": int(group["Documents"].sum()),
    "First year": int(group["Year"].min()),
    "Last year": int(group["Year"].max()),
    "Average publication year": round(float((group["Year"] * group["Documents"]).sum() / group["Documents"].sum()), 2),
    "Active years": int(group["Year"].nunique())
}), include_groups=False).reset_index().sort_values("Total documents", ascending=False)
save_csv(keyword_year, OUTPUT_DIR / "05_keyword_counts_by_year.csv")
save_csv(keyword_summary, OUTPUT_DIR / "05_keyword_temporal_summary.csv")
display(keyword_summary.head(40))


## 4. Términos emergentes

In [ ]:

complete_years = sorted(set(annual.loc[annual["Year status"] == "complete", "Year"]))
latest = max(complete_years)
window = int(CONFIG.get("recent_window_years", 3))
recent_start = latest - window + 1
previous_end = recent_start - 1
previous_start = previous_end - window + 1
recent = keyword_year[(keyword_year["Year"] >= recent_start) & (keyword_year["Year"] <= latest)].groupby("Keyword normalized")["Documents"].sum()
previous = keyword_year[(keyword_year["Year"] >= previous_start) & (keyword_year["Year"] <= previous_end)].groupby("Keyword normalized")["Documents"].sum()
emerging = pd.concat([recent.rename("Recent documents"), previous.rename("Previous documents")], axis=1).fillna(0)
emerging["Growth"] = emerging["Recent documents"] - emerging["Previous documents"]
emerging["Growth ratio"] = ((emerging["Recent documents"] + 1) / (emerging["Previous documents"] + 1)).round(3)
emerging = emerging.reset_index().sort_values(["Growth", "Recent documents"], ascending=[False, False])
save_csv(emerging, OUTPUT_DIR / "05_emerging_keywords.csv")
print("Recent window:", recent_start, "-", latest)
print("Previous window:", previous_start, "-", previous_end)
display(emerging.head(50))


## 5. Evolución de familias temáticas

In [ ]:

theme_docs["Year"] = pd.to_numeric(theme_docs["Year"], errors="coerce")
theme_year = theme_docs.dropna(subset=["Year"]).drop_duplicates(["EID", "Family", "Concept"]).groupby(["Family", "Concept", "Year"])["EID"].nunique().rename("Documents").reset_index()
save_csv(theme_year, OUTPUT_DIR / "05_theme_counts_by_year.csv")
text = (
    "Temporal analyses excluded configured partial or future years from growth comparisons. Keyword emergence was estimated by "
    "comparing two adjacent multi-year windows and should be interpreted as a descriptive trend rather than a forecast."
)
(REPORTS_DIR / "05_temporal_methodology.txt").write_text(text, encoding="utf-8")
print(text)
